# Train Transfer Learning Notebook
# Section: Title and description

"""
Waste Segregation — Transfer Learning Notebook

This notebook demonstrates transfer learning using a pretrained model (EfficientNet/MobileNet variants via timm).
Steps:
1. Setup and imports
2. Configure parameters
3. Helper functions (model build, freeze/unfreeze, evaluate)
4. Load dataset and inspect classes/counts
5. Train: freeze head then fine-tune backbone
6. Evaluate on test set

Run cells sequentially. Ensure dependencies from `requirements.txt` are installed.
"""


In [1]:
# Setup: installs (commented) and imports
# If you need to install packages in this environment, uncomment and run the pip install lines.

# !pip install -r requirements.txt

import os
import sys
from pathlib import Path
import time
import json

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ensure repo src is importable
repo_root = Path('.').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import timm
from src.dataset import WasteDataset
from src.transforms import get_transforms

print('Torch version:', torch.__version__)
print('Timm version:', timm.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


g:\waste_segregation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch version: 2.9.1+cpu
Timm version: 1.0.22
Device: cpu


In [2]:
# Configuration
data_dir = 'data/processed'
model_name = 'efficientnet_b0'   # change to 'mobilenetv3_large_100' or other timm model
img_size = 224
batch_size = 32
epochs = 12
freeze_epochs = 3
lr = 1e-3
weight_decay = 1e-4
workers = 4
out_dir = Path('models')
out_dir.mkdir(parents=True, exist_ok=True)

print('Config:')
print(dict(data_dir=data_dir, model_name=model_name, img_size=img_size, batch_size=batch_size, epochs=epochs, freeze_epochs=freeze_epochs, lr=lr))


Config:
{'data_dir': 'data/processed', 'model_name': 'efficientnet_b0', 'img_size': 224, 'batch_size': 32, 'epochs': 12, 'freeze_epochs': 3, 'lr': 0.001}


In [3]:
# Helper functions: model build, freeze/unfreeze, head params, evaluate

def build_model(name, num_classes, pretrained=True):
    model = timm.create_model(name, pretrained=pretrained, num_classes=num_classes)
    return model


def set_requires_grad(module, req):
    for p in module.parameters():
        p.requires_grad = req


def get_head_params(model):
    # attempt to locate common classifier heads
    for attr in ('classifier', 'fc', 'head'):
        if hasattr(model, attr):
            head = getattr(model, attr)
            return list(head.parameters())
    # fallback: parameters that require grad
    return [p for p in model.parameters() if p.requires_grad]


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total if total > 0 else 0.0

print('Helper functions defined')


Helper functions defined


In [4]:
# Load datasets and show class counts
from collections import Counter

train_ds = WasteDataset(data_dir, split='train', transform=get_transforms('torchvision','train', img_size))
val_ds = WasteDataset(data_dir, split='val', transform=get_transforms('torchvision','val', img_size))
test_ds = WasteDataset(data_dir, split='test', transform=get_transforms('torchvision','val', img_size))

print('Classes:', train_ds.classes)

# counts
def counts(ds):
    c = Counter()
    for _, lbl in ds.samples:
        c[train_ds.classes[lbl]] += 1
    return c

print('Train counts:', counts(train_ds))
print('Val counts:', counts(val_ds))
print('Test counts:', counts(test_ds))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)

print('Data loaders ready. Sample batch shapes:')
xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)


Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Train counts: Counter({'paper': 415, 'glass': 350, 'plastic': 336, 'metal': 286, 'cardboard': 281, 'trash': 95})
Val counts: Counter({'paper': 89, 'glass': 75, 'plastic': 73, 'metal': 62, 'cardboard': 61, 'trash': 21})
Test counts: Counter({'paper': 90, 'glass': 76, 'plastic': 73, 'metal': 62, 'cardboard': 61, 'trash': 21})
Data loaders ready. Sample batch shapes:


g:\waste_segregation\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


torch.Size([32, 3, 224, 224]) torch.Size([32])


In [5]:
# Training loop: freeze head, then fine-tune backbone

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(train_ds.classes)
model = build_model(model_name, num_classes=num_classes, pretrained=True)
model.to(device)

# Freeze all
set_requires_grad(model, False)
# enable head params
head_params = get_head_params(model)
for p in head_params:
    p.requires_grad = True

opt = torch.optim.Adam(head_params, lr=lr, weight_decay=weight_decay)
criterion = nn.CrossEntropyLoss()

best_val = 0.0
for epoch in range(1, epochs+1):
    is_finetune = epoch > freeze_epochs
    if is_finetune and (opt.param_groups[0]['lr'] == lr):
        print('Unfreezing backbone and switching to smaller LR for fine-tune')
        set_requires_grad(model, True)
        opt = torch.optim.Adam(model.parameters(), lr=lr*0.1, weight_decay=weight_decay)

    model.train()
    running_loss = 0.0
    t0 = time.time()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        opt.step()
        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_acc = evaluate(model, val_loader, device)
    print(f'Epoch {epoch}/{epochs}  loss={train_loss:.4f}  val_acc={val_acc:.4f}  time={(time.time()-t0):.1f}s')

    if val_acc > best_val:
        best_val = val_acc
        ckpt = out_dir / f'best_{model_name}.pth'
        torch.save({'model_state_dict': model.state_dict(), 'classes': train_ds.classes}, ckpt)
        print('Saved best checkpoint to', ckpt)

print('Training finished. Best val acc:', best_val)


Epoch 1/12  loss=3.2934  val_acc=0.3307  time=186.8s
Saved best checkpoint to models\best_efficientnet_b0.pth
Epoch 2/12  loss=2.2463  val_acc=0.4016  time=138.6s
Saved best checkpoint to models\best_efficientnet_b0.pth
Epoch 3/12  loss=1.7668  val_acc=0.4724  time=149.3s
Saved best checkpoint to models\best_efficientnet_b0.pth
Unfreezing backbone and switching to smaller LR for fine-tune
Epoch 4/12  loss=1.0672  val_acc=0.7270  time=583.0s
Saved best checkpoint to models\best_efficientnet_b0.pth
Epoch 5/12  loss=0.4300  val_acc=0.7743  time=374.3s
Saved best checkpoint to models\best_efficientnet_b0.pth
Epoch 6/12  loss=0.2843  val_acc=0.7874  time=362.1s
Saved best checkpoint to models\best_efficientnet_b0.pth
Epoch 7/12  loss=0.1906  val_acc=0.7900  time=368.6s
Saved best checkpoint to models\best_efficientnet_b0.pth
Epoch 8/12  loss=0.1631  val_acc=0.7900  time=344.6s
Epoch 9/12  loss=0.1331  val_acc=0.8058  time=355.6s
Saved best checkpoint to models\best_efficientnet_b0.pth
Epoch

In [6]:
# Final evaluation on test set
from pathlib import Path

# load best checkpoint if exists
best_ckpt = out_dir / f'best_{model_name}.pth'
if best_ckpt.exists():
    print('Loading best checkpoint', best_ckpt)
    ck = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ck['model_state_dict'])

test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=workers)
test_acc = evaluate(model, test_loader, device)
print('Test accuracy:', test_acc)


Loading best checkpoint models\best_efficientnet_b0.pth
Test accuracy: 0.8120104438642297


# Tips and next steps (markdown)
"""
Tips:
- To monitor training use TensorBoard: add tensorboard logging in the loop (SummaryWriter).
- For better results use `timm` EfficientNet/ResNet variants and consider learning rate schedulers.
- For mobile or edge deployment, export to ONNX/TorchScript and/or convert to TFLite with quantization.

Run in Colab / GPU:
- Ensure GPU runtime enabled.
- Install packages via pip: `pip install -r requirements.txt`.

Next steps:
- Add TensorBoard/W&B logging.
- Add early stopping and LR scheduling.
- Generate Grad-CAM visualizations for qualitative analysis.
"""
